> 在线讲义：[第 28 章 · Flow matching 推导：概率路径、条件目标为什么够用、rectified flow 与 π0 的动作头](https://wikieden.github.io/math-and-eval-stats/lessons/28-flow-matching.html) · 本书《数学与实验统计》

# 第 28 章 · Flow matching 推导：概率路径、条件目标为什么够用、rectified flow 与 π0 的动作头

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/28-flow-matching.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 速度场 | 条件均值 | `marginal_velocity(x, t)` | 3.2 − 1.2x（t = 0.5） |
| 第二步 回归 | 最小二乘 | `regress_cfm()` | 3.2021 − 1.2020x |
| 第三步 采样 | Euler | `euler_sample(1)` | 标准差 0 |
| 第四步 reflow | 直线速度 | `euler_sample(1, reflow_velocity)` | 0.4970 |
| 第四步 配对 | E‖x₁ − x₀‖² | `path_energy("ot")` | 4.2706 |
| 第五步 π0 | Beta 时间 | `pi0_time_mean()` | 0.4006 |

例子：沿用第 17 章的线性插值路径 x_t = (1 − t)·x_0 + t·x_1，x_0 ~ N(0, 1)（噪声），x_1 ~ N(2, 0.5²)（数据）。
两端都是高斯，边际速度场有闭式，可以拿来检查回归、采样和 reflow。
里程碑 ME-M21 的数值验证（前向闭式、Tweedie、OU 过程）不在这里。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
M1, S1 = 2.0, 0.5

**函数 `marginal_velocity`**：u_t(x) = E[x_1 − x_0 | x_t = x]。(x_1 − x_0, x_t) 联合高斯：

In [ ]:
def marginal_velocity(x, t: float):
    """u_t(x) = E[x_1 − x_0 | x_t = x]。(x_1 − x_0, x_t) 联合高斯：
    E[x_1 − x_0] = m，Var(x_t) = (1 − t)² + t² s²，Cov(x_1 − x_0, x_t) = t s² − (1 − t)。"""
    var = (1 - t) ** 2 + t * t * S1 ** 2
    cov = t * S1 ** 2 - (1 - t)
    return M1 + cov / var * (x - t * M1)

**函数 `regress_cfm`**：在条件目标 E‖v(x_t) − (x_1 − x_0)‖² 下，最优 v 是条件期望。用线性函数 v = α + β x 拟合样本，

In [ ]:
def regress_cfm(t: float = 0.5, n: int = 200000, seed: int = 0) -> tuple[float, float, float, float]:
    """在条件目标 E‖v(x_t) − (x_1 − x_0)‖² 下，最优 v 是条件期望。用线性函数 v = α + β x 拟合样本，
    和闭式比。返回 (拟合 α, 拟合 β, 闭式 α, 闭式 β)。"""
    rng = np.random.default_rng(seed)
    x0 = rng.standard_normal(n)
    x1 = M1 + S1 * rng.standard_normal(n)
    xt = (1 - t) * x0 + t * x1
    beta, alpha = np.polyfit(xt, x1 - x0, 1)
    var = (1 - t) ** 2 + t * t * S1 ** 2
    cov = t * S1 ** 2 - (1 - t)
    b = cov / var
    return float(alpha), float(beta), M1 - b * t * M1, b

**函数 `euler_sample`**：从 x_0 ~ N(0, 1) 用 Euler 积分 dx/dt = v(x, t) 到 t = 1，返回 (均值, 标准差)。

In [ ]:
def euler_sample(steps: int, vel=marginal_velocity, n: int = 20000, seed: int = 1) -> tuple[float, float]:
    """从 x_0 ~ N(0, 1) 用 Euler 积分 dx/dt = v(x, t) 到 t = 1，返回 (均值, 标准差)。"""
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(n)
    for i in range(steps):
        x = x + vel(x, i / steps) / steps
    return float(x.mean()), float(x.std())

**函数 `reflow_velocity`**：把 ODE 的起点、终点重新配对（x_0 ↔ m + s·x_0，一维时 ODE 映射是单调的这一条），新路径是直线，

In [ ]:
def reflow_velocity(x, t: float):
    """把 ODE 的起点、终点重新配对（x_0 ↔ m + s·x_0，一维时 ODE 映射是单调的这一条），新路径是直线，
    速度 = x_1 − x_0 = m + (s − 1)x_0，沿直线不变；用 x_t 反解出 x_0 = (x − t m)/(1 + t(s − 1))。"""
    x0 = (x - t * M1) / (1 + t * (S1 - 1))
    return M1 + (S1 - 1) * x0

**函数 `path_energy`**：E‖x_1 − x_0‖²：独立配对 vs 排序配对（一维的最优传输配对）。

In [ ]:
def path_energy(coupling: str, n: int = 200000, seed: int = 2) -> float:
    """E‖x_1 − x_0‖²：独立配对 vs 排序配对（一维的最优传输配对）。"""
    rng = np.random.default_rng(seed)
    x0 = rng.standard_normal(n)
    x1 = M1 + S1 * rng.standard_normal(n)
    if coupling == "ot":
        x0, x1 = np.sort(x0), np.sort(x1)
    return float(((x1 - x0) ** 2).mean())

**函数 `pi0_time_mean`**：π0 训练时的时间采样：(s − τ)/s ~ Beta(1.5, 1)，即偏向 τ 小（噪声多）的一端。返回 (E[τ], P(τ < 0.5))。

In [ ]:
def pi0_time_mean(s: float = 0.999, n: int = 400000, seed: int = 3) -> tuple[float, float]:
    """π0 训练时的时间采样：(s − τ)/s ~ Beta(1.5, 1)，即偏向 τ 小（噪声多）的一端。返回 (E[τ], P(τ < 0.5))。"""
    rng = np.random.default_rng(seed)
    tau = s * (1 - rng.beta(1.5, 1.0, n))
    return float(tau.mean()), float((tau < 0.5).mean())

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 概率路径与边际速度场")
for t in (0.0, 0.5, 0.9):
    b = (t * S1 ** 2 - (1 - t)) / ((1 - t) ** 2 + t * t * S1 ** 2)
    print(f"  t = {t}：u_t(x) = {M1 - b * t * M1:.4f} {"+" if b >= 0 else "−"} {abs(b):.4f}·x")

In [ ]:
print("\n第二步 条件目标的最优解就是条件期望：回归样本得到的就是边际速度场")
a, b, ea, eb = regress_cfm()
print(f"  t = 0.5：拟合 v = {a:.4f} − {-b:.4f}·x；闭式 {ea:.4f} − {-eb:.4f}·x")

In [ ]:
print("\n第三步 用 Euler 采样：边际路径是弯的，步数少会偏")
for k in (1, 2, 10, 100):
    m, s = euler_sample(k)
    print(f"  {k} 步：均值 {m:.4f}、标准差 {s:.4f}（目标 2、0.5）")

In [ ]:
print("\n第四步 Rectified flow：重新配对后路径是直线，1 步就准")
for k in (1, 2):
    m, s = euler_sample(k, reflow_velocity)
    print(f"  reflow 后 {k} 步：均值 {m:.4f}、标准差 {s:.4f}")
print(f"  E‖x_1 − x_0‖²：独立配对 {path_energy('ind'):.4f}，排序（最优传输）配对 {path_energy('ot'):.4f}")

In [ ]:
print("\n第五步 π0 的动作头：时间采样偏向噪声多的一端")
mt, pl = pi0_time_mean()
print(f"  τ = s·(1 − Beta(1.5, 1))：E[τ] = {mt:.4f}，P(τ < 0.5) = {pl:.4f}（均匀采样是 0.5、0.5）")

## 3. 代码里才看得到的

1. **reflow 速度要从 x_t 反解起点**：直线上 $x_t = (1 + t(s - 1))x_0 + tm$，代码先解出 x₀ 再给速度 m + (s − 1)x₀。
2. **"1 步标准差 0"是精确的 0**：u₀(x) = 2 − x，一步之后每个样本都恰好落在 2。
3. **起点样本带自己的抽样误差**：`euler_sample` 用固定种子的 2 万个起点，它们的均值、标准差和 0、1 差一点，所以多步采样的结果也差一点。

## 4. 自己改一改

1. 把 `S1` 改成 1（数据和噪声一样宽，只差一个平移），看 u_t(x) 变成什么、1 步 Euler 准不准。
2. 用 Heun（第 22 章）代替 Euler，比较 2 步时的标准差。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 一对样本连直线，平均起来是边际速度；
> 回归的最优是条件期望，条件目标就够用；
> 流线是弯的，步少抄近路；
> reflow 重新配对拉直线，和最优传输一个想法；
> π0 动作头：噪声连动作、回归速度、十步 Euler。